<a href="https://colab.research.google.com/github/keksenia/cstati-event-analytics/blob/main/notebooks/03_portfolio_overview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

IDENTITY_DIR = Path("/content/data/interim_private/identity")
IDENTITY_DIR.mkdir(parents=True, exist_ok=True)

print(IDENTITY_DIR)

/content/data/interim_private/identity


In [2]:
from pathlib import Path

identity_path = Path("/content/data/interim_private/identity/identity_records_private.csv")
metadata_path = Path("/content/manual/event_metadata.csv")

print("identity_records_private.csv exists:", identity_path.exists())
print("event_metadata.csv exists:", metadata_path.exists())

identity_records_private.csv exists: False
event_metadata.csv exists: False


In [3]:
from pathlib import Path

MANUAL_DIR = Path("/content/manual")
MANUAL_DIR.mkdir(parents=True, exist_ok=True)

print(MANUAL_DIR)

/content/manual


# 03 — Portfolio Overview

Цель ноутбука — построить первый продуктовый overview портфеля мероприятий cstati.

В этом ноутбуке мы анализируем:

- размер event portfolio;
- количество участников по событиям;
- clean identity coverage;
- новых и повторных участников;
- repeat participation;
- event family performance;
- базовые event journeys;
- retention proxy на уровне первого события.

Основная идея: рассматривать cstati не как набор отдельных мероприятий, а как event-community product с acquisition, activation и retention-механиками.


In [4]:
from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path("/content")

MANUAL_DIR = PROJECT_ROOT / "manual"
IDENTITY_DIR = PROJECT_ROOT / "data" / "interim_private" / "identity"
PROCESSED_PUBLIC_DIR = PROJECT_ROOT / "data" / "processed_public"
FIGURES_DIR = PROJECT_ROOT / "docs" / "figures"

PROCESSED_PUBLIC_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

identity_path = IDENTITY_DIR / "identity_records_private.csv"
metadata_path = MANUAL_DIR / "event_metadata.csv"

print("identity_path exists:", identity_path.exists(), identity_path)
print("metadata_path exists:", metadata_path.exists(), metadata_path)



identity_path exists: False /content/data/interim_private/identity/identity_records_private.csv
metadata_path exists: False /content/manual/event_metadata.csv


In [5]:
if not identity_path.exists():
    raise FileNotFoundError(
        "Не найден identity_records_private.csv. "
        "Сначала запусти 02_identity_resolution.ipynb до ячейки сохранения private outputs."
    )

if not metadata_path.exists():
    raise FileNotFoundError(
        "Не найден manual/event_metadata.csv. "
        "Загрузи manual-файлы в /content/manual/."
    )


identity_records_df = pd.read_csv(identity_path, dtype=str)
event_metadata = pd.read_csv(metadata_path, dtype=str)

print("identity_records_df:", identity_records_df.shape)
print("event_metadata:", event_metadata.shape)

display(identity_records_df.head(3))
display(event_metadata.head())


FileNotFoundError: Не найден identity_records_private.csv. Сначала запусти 02_identity_resolution.ipynb до ячейки сохранения private outputs.

In [ ]:
required_identity_cols = [
    "source_file",
    "source_row_number",
    "inferred_event_name",
    "canonical_event_id",
    "canonical_event_name",
    "is_auxiliary_source",
    "identity_confidence",
    "identity_conflict_flag",
    "participant_hash_private",
    "has_any_identifier",
    "has_strong_identifier",
]

missing_identity_cols = [
    col for col in required_identity_cols
    if col not in identity_records_df.columns
]

if missing_identity_cols:
    raise ValueError(
        "В identity_records_private.csv не хватает колонок: "
        + ", ".join(missing_identity_cols)
        + ". Скорее всего, нужно заново запустить 02_identity_resolution.ipynb после добавления identity_conflict_flag."
    )


def to_bool_series(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "y"])
    )


for col in ["is_auxiliary_source", "identity_conflict_flag", "has_any_identifier", "has_strong_identifier"]:
    identity_records_df[col] = to_bool_series(identity_records_df[col])


event_metadata["event_year"] = pd.to_numeric(event_metadata["event_year"], errors="coerce")

season_order_map = {
    "winter": 1,
    "spring": 2,
    "summer": 3,
    "autumn": 4,
    "fall": 4,
    "unknown": 9,
}

event_metadata["event_season_order"] = (
    event_metadata["event_season"]
    .fillna("unknown")
    .str.lower()
    .map(season_order_map)
    .fillna(9)
)

event_metadata["metadata_order"] = np.arange(len(event_metadata))

event_metadata["event_order"] = (
    event_metadata["event_year"].fillna(9999) * 100
    + event_metadata["event_season_order"]
    + event_metadata["metadata_order"] / 1000
)

event_metadata_short = event_metadata.rename(
    columns={"event_id": "canonical_event_id"}
)

display(event_metadata_short[
    [
        "canonical_event_id",
        "event_name",
        "event_family",
        "event_year",
        "event_season",
        "event_type",
        "strategic_role",
        "event_order",
    ]
])


In [ ]:
event_identity_df = identity_records_df[
    identity_records_df["is_auxiliary_source"] == False
].copy()

clean_identity_df = identity_records_df[
    (identity_records_df["is_auxiliary_source"] == False)
    & (identity_records_df["identity_conflict_flag"] == False)
    & (identity_records_df["identity_confidence"].isin(["high", "medium"]))
    & (identity_records_df["participant_hash_private"].notna())
].copy()

event_identity_df = event_identity_df.merge(
    event_metadata_short,
    on="canonical_event_id",
    how="left",
)

clean_identity_df = clean_identity_df.merge(
    event_metadata_short,
    on="canonical_event_id",
    how="left",
)

portfolio_layer_summary = pd.DataFrame([
    {
        "layer": "event_level_full",
        "rows": len(event_identity_df),
        "unique_participants": event_identity_df["participant_hash_private"].nunique(dropna=True),
        "events": event_identity_df["canonical_event_id"].nunique(dropna=True),
    },
    {
        "layer": "clean_identity",
        "rows": len(clean_identity_df),
        "unique_participants": clean_identity_df["participant_hash_private"].nunique(dropna=True),
        "events": clean_identity_df["canonical_event_id"].nunique(dropna=True),
    },
])

display(portfolio_layer_summary)


In [ ]:
event_participation_full = (
    event_identity_df[
        event_identity_df["participant_hash_private"].notna()
    ]
    .drop_duplicates(["canonical_event_id", "participant_hash_private"])
    .copy()
)

event_participation_clean = (
    clean_identity_df[
        clean_identity_df["participant_hash_private"].notna()
    ]
    .drop_duplicates(["canonical_event_id", "participant_hash_private"])
    .copy()
)

print("event_participation_full:", event_participation_full.shape)
print("event_participation_clean:", event_participation_clean.shape)

display(event_participation_clean.head())


In [ ]:
event_full_metrics = (
    event_identity_df
    .groupby("canonical_event_id", dropna=False)
    .agg(
        raw_rows=("source_row_number", "count"),
        full_unique_participants=("participant_hash_private", "nunique"),
        rows_with_any_identifier=("has_any_identifier", "sum"),
        rows_with_strong_identifier=("has_strong_identifier", "sum"),
    )
    .reset_index()
)

event_clean_metrics = (
    event_participation_clean
    .groupby("canonical_event_id", dropna=False)
    .agg(
        clean_unique_participants=("participant_hash_private", "nunique"),
    )
    .reset_index()
)

event_metrics = (
    event_metadata_short
    .merge(event_full_metrics, on="canonical_event_id", how="left")
    .merge(event_clean_metrics, on="canonical_event_id", how="left")
)

for col in [
    "raw_rows",
    "full_unique_participants",
    "rows_with_any_identifier",
    "rows_with_strong_identifier",
    "clean_unique_participants",
]:
    event_metrics[col] = event_metrics[col].fillna(0).astype(int)

event_metrics["strong_identifier_share"] = np.where(
    event_metrics["raw_rows"] > 0,
    event_metrics["rows_with_strong_identifier"] / event_metrics["raw_rows"],
    np.nan,
)

event_metrics["clean_coverage_share"] = np.where(
    event_metrics["full_unique_participants"] > 0,
    event_metrics["clean_unique_participants"] / event_metrics["full_unique_participants"],
    np.nan,
)

event_metrics = event_metrics.sort_values("event_order")

display(event_metrics[
    [
        "canonical_event_id",
        "event_name",
        "event_family",
        "event_year",
        "event_type",
        "strategic_role",
        "raw_rows",
        "full_unique_participants",
        "clean_unique_participants",
        "strong_identifier_share",
        "clean_coverage_share",
    ]
])


In [ ]:
clean_sequence = event_participation_clean[
    [
        "participant_hash_private",
        "canonical_event_id",
        "event_name",
        "event_family",
        "event_year",
        "event_order",
    ]
].dropna(subset=["participant_hash_private", "canonical_event_id", "event_order"]).copy()

clean_sequence = clean_sequence.sort_values(
    ["participant_hash_private", "event_order", "canonical_event_id"]
)

clean_sequence["participant_event_rank"] = (
    clean_sequence
    .groupby("participant_hash_private")
    .cumcount()
    + 1
)

clean_sequence["is_first_event"] = clean_sequence["participant_event_rank"] == 1
clean_sequence["prior_events_cnt"] = clean_sequence["participant_event_rank"] - 1

new_repeat_metrics = (
    clean_sequence
    .groupby("canonical_event_id", dropna=False)
    .agg(
        clean_participants=("participant_hash_private", "nunique"),
        new_participants=("is_first_event", "sum"),
        repeat_participants=("prior_events_cnt", lambda s: (s > 0).sum()),
        avg_prior_events=("prior_events_cnt", "mean"),
    )
    .reset_index()
)

new_repeat_metrics["new_share"] = (
    new_repeat_metrics["new_participants"]
    / new_repeat_metrics["clean_participants"]
).round(3)

new_repeat_metrics["repeat_share"] = (
    new_repeat_metrics["repeat_participants"]
    / new_repeat_metrics["clean_participants"]
).round(3)

event_metrics = event_metrics.merge(
    new_repeat_metrics,
    on="canonical_event_id",
    how="left",
)

for col in [
    "clean_participants",
    "new_participants",
    "repeat_participants",
]:
    event_metrics[col] = event_metrics[col].fillna(0).astype(int)

event_metrics["new_share"] = event_metrics["new_share"].fillna(0)
event_metrics["repeat_share"] = event_metrics["repeat_share"].fillna(0)
event_metrics["avg_prior_events"] = event_metrics["avg_prior_events"].fillna(0)

display(event_metrics[
    [
        "event_name",
        "event_family",
        "event_year",
        "clean_participants",
        "new_participants",
        "repeat_participants",
        "new_share",
        "repeat_share",
        "avg_prior_events",
    ]
].sort_values("clean_participants", ascending=False))


In [ ]:
family_metrics = (
    event_metrics
    .groupby("event_family", dropna=False)
    .agg(
        events=("canonical_event_id", "nunique"),
        raw_rows=("raw_rows", "sum"),
        full_unique_participants=("full_unique_participants", "sum"),
        clean_unique_participants=("clean_unique_participants", "sum"),
        clean_participants=("clean_participants", "sum"),
        new_participants=("new_participants", "sum"),
        repeat_participants=("repeat_participants", "sum"),
        avg_strong_identifier_share=("strong_identifier_share", "mean"),
        avg_clean_coverage_share=("clean_coverage_share", "mean"),
    )
    .reset_index()
)

family_metrics["new_share"] = np.where(
    family_metrics["clean_participants"] > 0,
    family_metrics["new_participants"] / family_metrics["clean_participants"],
    np.nan,
).round(3)

family_metrics["repeat_share"] = np.where(
    family_metrics["clean_participants"] > 0,
    family_metrics["repeat_participants"] / family_metrics["clean_participants"],
    np.nan,
).round(3)

family_metrics = family_metrics.sort_values("clean_participants", ascending=False)

display(family_metrics)


In [ ]:
participant_depth = (
    clean_sequence
    .groupby("participant_hash_private")
    .agg(
        events_cnt=("canonical_event_id", "nunique"),
        first_event=("event_name", "first"),
        first_event_family=("event_family", "first"),
    )
    .reset_index()
)

depth_distribution = (
    participant_depth
    .groupby("events_cnt")
    .agg(
        participants=("participant_hash_private", "nunique")
    )
    .reset_index()
)

depth_distribution["participant_share"] = (
    depth_distribution["participants"]
    / depth_distribution["participants"].sum()
).round(3)

display(depth_distribution)


In [ ]:
first_event_retention = participant_depth.copy()
first_event_retention["returned_later"] = first_event_retention["events_cnt"] > 1

retention_by_first_family = (
    first_event_retention
    .groupby("first_event_family", dropna=False)
    .agg(
        first_time_participants=("participant_hash_private", "nunique"),
        returned_later=("returned_later", "sum"),
        avg_events_per_participant=("events_cnt", "mean"),
    )
    .reset_index()
)

retention_by_first_family["return_rate"] = (
    retention_by_first_family["returned_later"]
    / retention_by_first_family["first_time_participants"]
).round(3)

retention_by_first_family = retention_by_first_family.sort_values(
    "first_time_participants",
    ascending=False,
)

display(retention_by_first_family)


In [ ]:
journey_df = clean_sequence.sort_values(
    ["participant_hash_private", "event_order"]
).copy()

journey_df["next_event_name"] = (
    journey_df
    .groupby("participant_hash_private")["event_name"]
    .shift(-1)
)

journey_df["next_event_family"] = (
    journey_df
    .groupby("participant_hash_private")["event_family"]
    .shift(-1)
)

event_transitions = (
    journey_df[
        journey_df["next_event_name"].notna()
    ]
    .groupby(["event_name", "next_event_name"], dropna=False)
    .agg(
        participants=("participant_hash_private", "nunique")
    )
    .reset_index()
    .sort_values("participants", ascending=False)
)

family_transitions = (
    journey_df[
        journey_df["next_event_family"].notna()
    ]
    .groupby(["event_family", "next_event_family"], dropna=False)
    .agg(
        participants=("participant_hash_private", "nunique")
    )
    .reset_index()
    .sort_values("participants", ascending=False)
)

display(event_transitions.head(20))
display(family_transitions.head(20))


In [ ]:
# 1. Размер событий по clean participants

plot_df = event_metrics.sort_values("clean_unique_participants", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(plot_df["event_name"], plot_df["clean_unique_participants"])
ax.set_title("Clean participants by event")
ax.set_xlabel("Clean unique participants")
ax.set_ylabel("Event")

plt.tight_layout()
fig_path = FIGURES_DIR / "clean_participants_by_event.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()

print("saved:", fig_path)

# 2. New vs repeat share

plot_df = event_metrics.sort_values("event_order").copy()

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(plot_df["event_name"], plot_df["new_participants"], label="New participants")
ax.bar(
    plot_df["event_name"],
    plot_df["repeat_participants"],
    bottom=plot_df["new_participants"],
    label="Repeat participants",
)

ax.set_title("New vs repeat participants by event")
ax.set_xlabel("Event")
ax.set_ylabel("Participants")
ax.tick_params(axis="x", rotation=75)
ax.legend()

plt.tight_layout()
fig_path = FIGURES_DIR / "new_vs_repeat_by_event.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()

print("saved:", fig_path)

# 3. Portfolio health matrix

plot_df = event_metrics[
    event_metrics["clean_participants"] > 0
].copy()

fig, ax = plt.subplots(figsize=(9, 6))

sizes = plot_df["clean_participants"].clip(lower=10) * 2

ax.scatter(
    plot_df["new_share"],
    plot_df["repeat_share"],
    s=sizes,
    alpha=0.7,
)

for _, row in plot_df.iterrows():
    ax.annotate(
        row["event_name"],
        (row["new_share"], row["repeat_share"]),
        fontsize=8,
        alpha=0.8,
    )

ax.set_title("Portfolio health matrix")
ax.set_xlabel("New participants share")
ax.set_ylabel("Repeat participants share")

plt.tight_layout()
fig_path = FIGURES_DIR / "portfolio_health_matrix.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()

print("saved:", fig_path)

# 4. Repeat depth distribution

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(depth_distribution["events_cnt"], depth_distribution["participants"])

ax.set_title("Distribution of event depth per participant")
ax.set_xlabel("Number of events per participant")
ax.set_ylabel("Participants")

plt.tight_layout()
fig_path = FIGURES_DIR / "repeat_depth_distribution.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()

print("saved:", fig_path)


In [ ]:
event_scorecard = event_metrics[
    [
        "event_name",
        "event_family",
        "event_year",
        "strategic_role",
        "clean_participants",
        "new_participants",
        "repeat_participants",
        "new_share",
        "repeat_share",
        "strong_identifier_share",
        "clean_coverage_share",
    ]
].copy()

event_scorecard["acquisition_score"] = (
    event_scorecard["new_participants"].rank(pct=True)
    + event_scorecard["new_share"].rank(pct=True)
) / 2

event_scorecard["retention_score"] = (
    event_scorecard["repeat_participants"].rank(pct=True)
    + event_scorecard["repeat_share"].rank(pct=True)
) / 2

event_scorecard["data_quality_score"] = (
    event_scorecard["strong_identifier_share"].fillna(0).rank(pct=True)
    + event_scorecard["clean_coverage_share"].fillna(0).rank(pct=True)
) / 2

event_scorecard["portfolio_score"] = (
    0.4 * event_scorecard["acquisition_score"]
    + 0.4 * event_scorecard["retention_score"]
    + 0.2 * event_scorecard["data_quality_score"]
)

event_scorecard = event_scorecard.sort_values("portfolio_score", ascending=False)

display(event_scorecard)


In [ ]:
public_outputs = {
    "metrics_event_level": event_metrics,
    "metrics_family_level": family_metrics,
    "metrics_repeat_depth": depth_distribution,
    "metrics_first_event_retention": retention_by_first_family,
    "event_transitions": event_transitions,
    "family_transitions": family_transitions,
    "event_scorecard": event_scorecard,
}

for name, df in public_outputs.items():
    output_path = PROCESSED_PUBLIC_DIR / f"{name}.parquet"

    try:
        df.to_parquet(output_path, index=False)
        print("saved:", output_path)
    except Exception as e:
        fallback_path = PROCESSED_PUBLIC_DIR / f"{name}.csv"
        df.to_csv(fallback_path, index=False)
        print("parquet failed, saved csv instead:", fallback_path, "|", e)


In [ ]:
print("portfolio summary")
display(portfolio_layer_summary)

print("\nevent metrics")
display(event_metrics[
    [
        "event_name",
        "event_family",
        "clean_participants",
        "new_participants",
        "repeat_participants",
        "new_share",
        "repeat_share",
        "clean_coverage_share",
    ]
].sort_values("clean_participants", ascending=False))

print("\nfamily metrics")
display(family_metrics)

print("\nretention by first family")
display(retention_by_first_family)

print("\ntop transitions")
display(event_transitions.head(10))

print("\nevent scorecard")
display(event_scorecard.head(10))


## Итоги portfolio overview

В этом ноутбуке построен первый продуктовый overview портфеля мероприятий cstati.

### Ключевые результаты

- В full event-level layer попало 18 событий, 5 257 event-level строк и 3 636 уникальных participant keys.
- В clean identity layer попало 16 событий, 2 933 строки и 2 527 уникальных participant keys.
- Самые крупные acquisition/reach события в clean layer: `Нейрорейв`, `Посвят'25`, `Посвят'23`.
- Самые сильные retention/community события по repeat-share: `Настолки'24`, `Антипосвят'25`, `Поход'25`.
- Большинство участников в clean layer посетили только одно событие: 90.3%.
- 9.7% участников посетили два и более события.
- Самый сильный event-to-event переход: `Поход'24 → Поход'25`.
- На уровне product scorecard в топ вышли `CSFEST'25`, `Поход'25`, `Поход'24`, `Посвят'25`.

### Продуктовая интерпретация

Портфель мероприятий работает как двухконтурная система:

1. **Acquisition-контур**: массовые события привлекают новую аудиторию.
   - `Нейрорейв`
   - `Посвят`
   - `CSFEST`

2. **Retention/community-контур**: более камерные или повторяемые события возвращают участников.
   - `Поход`
   - `Антипосвят`
   - `Настолки`
   - часть `Экватор`

Главный продуктовый вывод: cstati не стоит оценивать мероприятия только по размеру аудитории. Массовые события дают reach и acquisition, но retention-сигнал сильнее проявляется в событиях с более выраженной community-механикой.

### Ограничения

- Retention и journeys пока считаются как proxy, потому что точные `event_date` ещё не заполнены для всех событий.
- Порядок событий строится по `event_year`, `event_season` и ручному порядку в metadata.
- `Бал ФКН'24` и `Коллаб'24` требуют отдельной обработки перед использованием в clean retention metrics.
- Clean layer намеренно строгий: конфликтные identity cases и low-confidence строки исключены из чувствительных метрик.

### Следующий шаг

Следующий ноутбук — `04_deep_dive_events.ipynb`.

В нём нужно разобрать 4–6 ключевых событий глубже: funnel, audience mix, data quality, repeat potential и actionable recommendations.

In [ ]:
print("=== INPUTS ===")
print("identity_records_df:", identity_records_df.shape)
print("event_metadata:", event_metadata.shape)

print("\n=== LAYERS ===")
display(portfolio_layer_summary)

print("\n=== EVENT METRICS CHECK ===")
print("events in metadata:", event_metadata["event_id"].nunique())
print("events in event_metrics:", event_metrics["canonical_event_id"].nunique())
print("events with clean participants > 0:", (event_metrics["clean_participants"] > 0).sum())
print("total clean participants rows:", event_metrics["clean_participants"].sum())
print("unique clean participants:", clean_sequence["participant_hash_private"].nunique())

display(
    event_metrics[
        [
            "event_name",
            "event_family",
            "event_year",
            "clean_participants",
            "new_participants",
            "repeat_participants",
            "new_share",
            "repeat_share",
            "clean_coverage_share",
        ]
    ].sort_values("clean_participants", ascending=False)
)

print("\n=== FAMILY METRICS ===")
display(family_metrics)

print("\n=== REPEAT DEPTH ===")
display(depth_distribution)

print("\n=== FIRST EVENT RETENTION PROXY ===")
display(retention_by_first_family)

print("\n=== TOP EVENT TRANSITIONS ===")
display(event_transitions.head(15))

print("\n=== TOP FAMILY TRANSITIONS ===")
display(family_transitions.head(15))

print("\n=== EVENT SCORECARD TOP 10 ===")
display(event_scorecard.head(10))

In [ ]:
display(event_metrics[
    [
        "event_name",
        "event_family",
        "clean_participants",
        "new_participants",
        "repeat_participants",
        "new_share",
        "repeat_share",
        "clean_coverage_share",
    ]
].sort_values("clean_participants", ascending=False))

display(family_metrics)

display(depth_distribution)

display(retention_by_first_family)

display(event_transitions.head(15))

display(event_scorecard.head(10))